In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install torch torchvision thop pandas --quiet

In [3]:
!git clone https://github.com/DingXiaoH/RepVGG.git
import sys
sys.path.append('/content/RepVGG')

Cloning into 'RepVGG'...
remote: Enumerating objects: 581, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 581 (delta 191), reused 174 (delta 138), pack-reused 345 (from 1)
Receiving objects: 100% (581/581), 485.19 KiB | 2.35 MiB/s, done.
Resolving deltas: 100% (347/347), done.


In [5]:
from repvgg import create_RepVGG_A0, create_RepVGG_B0, create_RepVGG_B1g2, repvgg_model_convert

import torch
import torchvision.models as models
import time
import os
from thop import profile
import pandas as pd

DEVICE_GPU = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_CPU = torch.device("cpu")

BATCH_SIZE = 1
IMG_SIZE = 224
REPEATS = 50


# Models
REP_MODELS = {
    "RepVGG-A0": create_RepVGG_A0,
    "RepVGG-B0": create_RepVGG_B0,
    "RepVGG-B1g2": create_RepVGG_B1g2
}

RESNET_MODELS = {
    "ResNet-18": lambda: models.resnet18(pretrained=True),
    "ResNet-34": lambda: models.resnet34(pretrained=True),
    "ResNet-50": lambda: models.resnet50(pretrained=True)
}

# Pretrained paths (you must upload to Colab)
PRETRAINED_PATHS = {
    "RepVGG-A0": "/content/RepVGG/pretrained/RepVGG-A0-train.pth",
    "RepVGG-B0": "/content/RepVGG/pretrained/RepVGG-B0-train.pth",
    "RepVGG-B1g2": "/content/RepVGG/pretrained/RepVGG-B1g2-train.pth"
}


def get_model_params(model):
    return sum(p.numel() for p in model.parameters())

def get_model_size(model, filename="temp.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def get_model_flops(model, input_size=(1,3,224,224)):
    input_tensor = torch.randn(input_size)
    flops, _ = profile(model, inputs=(input_tensor,), verbose=False)
    return flops

def measure_latency(model, device, input_size=(1,3,224,224), repeats=50):
    model.eval()
    input_tensor = torch.randn(input_size).to(device)
    model.to(device)

    # GPU warm-up
    if device.type == "cuda":
        for _ in range(10):
            _ = model(input_tensor)
        torch.cuda.synchronize()

    start_time = time.time()
    for _ in range(repeats):
        _ = model(input_tensor)
    if device.type == "cuda":
        torch.cuda.synchronize()
    end_time = time.time()
    return (end_time - start_time)/repeats

def benchmark_model(name, model_fn, pretrained_path=None, deploy=False):
    print(f"\n===== {name} | Deploy={deploy} =====")

    if "RepVGG" in name:
        model = model_fn(deploy=False)
        if pretrained_path:
            model.load_state_dict(torch.load(pretrained_path, map_location='cpu'))
        if deploy:
            model = repvgg_model_convert(model)
    else:
        model = model_fn()

    # Metrics
    params = get_model_params(model)
    size_mb = get_model_size(model)
    flops = get_model_flops(model)

    cpu_lat = measure_latency(model, DEVICE_CPU, repeats=REPEATS)
    gpu_lat = measure_latency(model, DEVICE_GPU, repeats=REPEATS) if torch.cuda.is_available() else None

    print(f"Params: {params/1e6:.2f} M")
    print(f"Model size: {size_mb:.2f} MB")
    print(f"FLOPs: {flops/1e9:.2f} G")
    print(f"CPU Latency: {cpu_lat*1000:.2f} ms")
    if gpu_lat:
        print(f"GPU Latency: {gpu_lat*1000:.2f} ms")

    return {
        "name": name,
        "deploy": deploy,
        "params": params,
        "size_mb": size_mb,
        "flops": flops,
        "cpu_lat": cpu_lat,
        "gpu_lat": gpu_lat
    }

results = []

# RepVGG train + deploy
for name, fn in REP_MODELS.items():
    pretrained = PRETRAINED_PATHS[name]
    results.append(benchmark_model(name, fn, pretrained_path=pretrained, deploy=False))
    results.append(benchmark_model(name, fn, pretrained_path=pretrained, deploy=True))

# ResNet baselines
for name, fn in RESNET_MODELS.items():
    results.append(benchmark_model(name, fn))

df = pd.DataFrame(results)
df.to_csv("repvgg_colab_benchmark_results.csv", index=False)
print("\n✅ Benchmark complete. Results saved to repvgg_colab_benchmark_results.csv")


===== RepVGG-A0 | Deploy=False =====
RepVGG Block, identity =  None
RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  None
RepVGG Block, identity =  BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
RepVGG Block, identity =  BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_runnin

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Params: 41.36 M
Model size: 165.46 MB
FLOPs: 8.81 G
CPU Latency: 231.36 ms
GPU Latency: 6.90 ms

===== ResNet-18 | Deploy=False =====
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Params: 11.69 M
Model size: 46.83 MB
FLOPs: 1.82 G
CPU Latency: 60.98 ms
GPU Latency: 3.85 ms

===== ResNet-34 | Deploy=False =====


...

Params: 21.80 M
Model size: 87.32 MB
FLOPs: 3.68 G
CPU Latency: 108.52 ms
GPU Latency: 4.86 ms

===== ResNet-50 | Deploy=False =====


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 199MB/s]


Params: 25.56 M
Model size: 102.52 MB
FLOPs: 4.13 G
CPU Latency: 146.44 ms
GPU Latency: 6.93 ms

✅ Benchmark complete. Results saved to repvgg_colab_benchmark_results.csv
